# Qwen3.8-27B reference run on a local / rented NVIDIA GPU · public_test with Gemini transcripts (v3, llama.cpp)

The same run as `colab_qwen38_27b_v3.ipynb` (same agent, prompt, examples, rules and settings), for a PC without an A100:
- **Model**: a quantized GGUF of Qwen3.8-27B served by **llama.cpp** (`llama-server`), instead of the FP8 weights in vLLM. The quantization is picked from the GPU memory (table below). Prompts are still built with Qwen's own tokenizer and chat template and sent as token ids, so thinking, the thinking budget and the closing of unfinished thoughts work as in the Colab notebook.
- **Files**: a local folder (`DRIVE`, default `ZOLA_data` next to this notebook) instead of Google Drive.
- **No 5-hour limit**: `HOURS` is 72. A stopped or crashed run continues where it stopped when you run all again.
- Works on Windows and Linux. Needs an NVIDIA driver recent enough for CUDA 12 (Windows 551+, Linux 570+), about 35 GB of free disk, Python 3.10+.

| GPU memory | `QUANT` picked | Slots | Note |
|---|---|---|---|
| 44 GB+ | Q8_0 (27 GB) | 4 | same precision class as the Colab FP8 run |
| 30–43 GB (RTX 5090) | Q6_K (22 GB) | 2 | very close to Q8 |
| 22.5–29 GB (RTX 3090 / 4090) | Q4_K_M (16 GB) | 2 | **recommended minimum**; small quality loss expected |
| 19–22 GB | Q4_K_M | 1 | half the speed |
| 15–18 GB (RTX 4060 Ti 16 GB, 4080) | Q3_K_M (12.5 GB) | 1 | visibly weaker model |
| 11–14 GB (RTX 3060 12 GB) | IQ2_M (9.8 GB) | 1 | heavily degraded; expect a clearly lower score than the Colab run |

A 27B model does not fit a 12 GB card at a useful precision. If you can choose the machine, take **24 GB or more**.
Every quantized run is a different model from the 70.88 run, so compare scores with that in mind.

**History**: v1 64.55 public LB, v2 70.88 (both Colab, FP8). v3 changes (examples from train gold, argument habits, general rules, second transcript, no "list everything", train check) are described in `colab_qwen38_27b_v3.ipynb` and CLAUDE.md.

**Reference only, never the final solution.** 27B breaks the 5B LLM limit, and the final-request text comes from Gemini (an API). Never distill from its public traces: that is labelling test data.

**Setup**
1. Install Jupyter if the machine has none: `pip install jupyterlab`, then `jupyter lab` in this notebook's folder (VS Code also works).
2. The three input files `train+public_test.zip`, `raw_asr_public.json` (Gemini text) and `asr_transcripts.json` (01c output, every clip): either in a folder `ZOLA_data` next to this notebook, or on Google Drive with `SYNC` set (next section).
3. Optional but faster: a Hugging Face access token (huggingface.co → Settings → Access Tokens → type **Read**). Either set the environment variable `HF_TOKEN` before starting Jupyter, or save the token alone in a text file `hf_token.txt` in `ZOLA_data` (or in the `SYNC` folder on Google Drive, from where it is fetched). Never paste it into the notebook: the notebook is shared, the file is not. The notebook checks the token, never prints it, and downloads without it if Hugging Face rejects it.
4. Run all. The first run downloads llama.cpp (about 0.7 GB) and the GGUF (10–27 GB) into `ZOLA_data/`.

**Google Drive (`SYNC`, optional)**: the run always works in the local folder `DRIVE`. With `SYNC` set it first fetches from Google Drive the input files `DRIVE` lacks and the output folders of earlier runs (newer files only), so a run stopped on one machine continues on another; then it copies the output folders back to Google Drive every `SYNC_MIN` minutes and at the end, also after a crash. llama.cpp, the GGUF and the unzipped data never go to Google Drive. Two ways:
- **Google Drive for desktop** (Windows/macOS, signed in): `SYNC = "G:/My Drive/ZOLA_data"` (your drive letter and folder).
- **rclone** (any machine, also Linux without a desktop): `SYNC = "gdrive:ZOLA_data"`, which is `My Drive/ZOLA_data`, the Colab folder. The notebook downloads rclone if it is not installed, but the Google sign-in is yours to do once: on your own PC install rclone (`winget install Rclone.Rclone`), run `rclone config` → `n` (new remote) → name `gdrive` → storage `drive` → Enter for every other question → sign in in the browser. Then copy the config file (its path is printed by `rclone config file`) to the same place on the rented machine (Linux: `~/.config/rclone/rclone.conf`, Windows: `%APPDATA%\rclone\rclone.conf`), or point `RCLONE_CONFIG` at it. That file holds a Google Drive access token: delete it from the rented machine when you are done.

Do not set `DRIVE` itself to the Google Drive folder: the 10–27 GB model and thousands of unzipped files would be uploaded.

To continue the Colab v3 run's folder instead of starting a new one, set `OUT_NAME = "qwen38_out_v3"`; its finished samples are kept, so the submission then mixes FP8 and quantized answers.

**Output** in `ZOLA_data/qwen38_out_v3_local/` (`traces.jsonl`, `submission.jsonl`) and `qwen38_out_v3_local_train/` (`traces.jsonl`, `scored.jsonl`). The server log is `ZOLA_data/llama-server.log`.

**If the server does not start** (the cell prints the end of its log): out of memory → `PARALLEL = 1`, then a smaller `QUANT`; "unknown model architecture" → a newer `LLAMA_TAG`; a missing CUDA library → update the NVIDIA driver, or set `LLAMA_SERVER` to a llama-server you installed yourself.

In [ ]:
import time

T0 = time.time()  # the clock for HOURS: downloads and server start-up count too

HOURS = 72.0  # whole notebook. No Colab limit here: lower it only if the machine is rented by the hour
MARGIN_MIN = 20  # no round or chunk starts this close to the end
DRIVE = "ZOLA_data"  # local working folder: inputs, outputs, downloads. Keep it on the local disk; use SYNC for Google Drive
SYNC = None  # Google Drive copy (see the top): "G:/My Drive/ZOLA_data" (Google Drive for desktop) or "gdrive:ZOLA_data" (rclone remote:folder)
SYNC_MIN = 10  # outputs are copied to SYNC at most this often while running, and always at the end
RCLONE = None  # path of rclone; None: from PATH, else the current build is downloaded into DRIVE/rclone
HF_TOKEN_FILE = "hf_token.txt"  # Hugging Face token for faster downloads: the HF_TOKEN environment variable, else this file in DRIVE (fetched from SYNC too). Never paste the token here
ZIP = f"{DRIVE}/train+public_test.zip"
GEMINI = f"{DRIVE}/raw_asr_public.json"  # Gemini text: keys "<split>/<clip path>", or sample ids with <id>_h<N> for history[N]
FILL = f"{DRIVE}/asr_transcripts.json"  # 01c output: every clip, used where Gemini has none and as the second transcript
RUN = ("public_test", "train")  # in this order on the one loaded model. "train": TRAIN_CHECK held-out train rows with gold text, scored against gold
TRAIN_CHECK = 82  # rows of lifemate's validation sets (unseen-app and seen-app) in the train check
OUT_NAME = "qwen38_out_v3_local"  # public_test here, train in <OUT_NAME>_train. Not the Colab folder: a quantized model is a different run
TOKENIZER_ID = "Qwen/Qwen3.8-27B-FP8"  # tokenizer and chat template only (a few MB), the same as the Colab run
GGUF_REPO = "bartowski/Qwen3.8-27B-GGUF"  # files Qwen3.8-27B-<QUANT>.gguf
QUANT = "auto"  # from the GPU memory (table above), or a name: "Q8_0", "Q6_K", "Q5_K_M", "Q4_K_M", "Q3_K_M", "IQ2_M"
PARALLEL = "auto"  # server slots = conversations generated at the same time; each slot holds a MAX_LEN context in GPU memory
GGUF = None  # path of a .gguf you already have; then nothing is downloaded and QUANT is ignored
VRAM_GB = None  # None: read from nvidia-smi (first GPU)
GPU_LAYERS = "all"  # "all": the server fails at start-up if the model does not fit. "auto": the rest goes to the CPU, far slower
LLAMA_TAG = "b12318"  # llama.cpp release to download
LLAMA_SERVER = None  # path of an existing llama-server(.exe); then no llama.cpp download
PORT = 8765
SERVER_ARGS = []  # extra llama-server arguments
START_MIN = 30  # give up if the server is not ready after this
THINK = True
EFFORT = "medium"
THINK_TOKENS = 2048  # per step; a longer thought is closed for the model, then the answer is generated
ANSWER_TOKENS = 320  # the longest gold call is 304 characters
MAX_STEPS = 8  # gold never uses more than 6 calls
CHUNK = None  # samples in work together. None: PARALLEL, so a sample's next step finds its prompt still cached in its slot
MAX_LEN = 16384  # the competition's context limit
SEED = 42
DEMOS = 5  # solved train examples per sample in the system prompt, most similar request first. 0 turns them off
DEMO_RESULT_CHARS = 240  # each tool result inside an example is cut here
FIRST_PROMPT_MAX = 10500  # tokens: examples are dropped, least similar first, until the first prompt fits (later steps add calls and results)
ARG_HINTS = True  # per installed tool: optional parameters train gold almost never / usually fills
ALT_ASR = True  # show the 01c transcript of the final request next to Gemini's where the two differ enough
ALT_BELOW = 0.8  # "enough": token F1 between the two below this (183 of 634 public requests; trivial differences such as "1" / "một" stay above)
LIST_ON_EMPTY = False  # v1/v2 listed everything after an empty search; train gold never does
# Appended to lifemate's system prompt (lifemate.py stays the copy 02 trains on). Every rule below is a habit of train gold.
EXTRA_RULES = """

Bổ sung:
- Mỗi lượt chỉ gọi MỘT tool. Sau mỗi KẾT QUẢ bạn sẽ được hỏi tiếp, nên việc cần nhiều tool thì trả về ngay lệnh đầu tiên, các lệnh sau gọi ở lượt sau.
- Lời người dùng là bản chép lại từ giọng nói, có thể nghe nhầm tên, số hoặc từ: hiểu theo ý gần nhất với tool và dữ liệu đang có, đừng vội kết luận là không làm được.
- Chỉ gọi sys.ask_missing_info cho giá trị người dùng không nói và cũng không tra hay suy ra được. Giá trị tính được từ hội thoại, kết quả tool hoặc bảng ngày thì tự tra, tự tính.
- Trước khi hỏi một ID (…_id), hãy gọi tool đọc để tìm nó. Chỉ hỏi khi đọc rồi mà không thấy hoặc không phân biệt được.
- Tool đọc: mặc định gọi KHÔNG kèm bộ lọc tuỳ chọn (loại, trạng thái, khoảng ngày, phòng, tầng, địa điểm...) rồi tự chọn dòng cần trong KẾT QUẢ. Chỉ điền tham số bắt buộc, và từ khoá tìm kiếm (query) khi phải tìm theo tên.
- Tool ghi: tham số tuỳ chọn kiểu nhãn, ghi chú, địa điểm, lặp lại, ưu tiên, cách thanh toán, địa chỉ (label, note, location, repeat, priority, payment, address...) thì KHÔNG tự điền từ ngữ cảnh. Chỉ điền khi người dùng yêu cầu đúng điều đó ("ghi chú là...", "đặt tên là...", "lặp lại hằng ngày").
- Tạo sự kiện lịch mà người dùng không nói giờ kết thúc: end = start + 1 giờ.
- Văn bản tự do (nội dung nhắc, tin nhắn, tiêu đề): dùng đúng cụm từ người dùng nói, bỏ từ đệm, không diễn đạt lại và không thêm chi tiết.
- Các "Ví dụ mẫu" ở cuối là lời giải chuẩn của những yêu cầu tương tự. Yêu cầu giống ví dụ thì làm đúng như ví dụ: cùng thứ tự tool, cùng bộ tham số, cùng cách viết giá trị. Ngày giờ, ID và dữ liệu thì lấy từ hội thoại hiện tại, không lấy từ ví dụ."""
ALT_NOTE = '\n[Bản chép thứ hai của cùng câu nói này, do máy khác nghe: "{}". Chỗ hai bản khác nhau thì chọn cách hiểu khớp với tool và dữ liệu đang có.]'

import os

import re
import shutil
import subprocess
from pathlib import Path

DRIVE = os.path.abspath(DRIVE)
os.makedirs(DRIVE, exist_ok=True)
ZIP, GEMINI, FILL = (os.path.join(DRIVE, os.path.basename(p)) for p in (ZIP, GEMINI, FILL))
TOKEN_PATH = os.path.join(DRIVE, HF_TOKEN_FILE)
INPUTS = (ZIP, GEMINI, FILL, TOKEN_PATH)  # what pull() fetches from SYNC when missing here; the token file is optional
os.environ.setdefault("HF_XET_HIGH_PERFORMANCE", "1")  # hf_xet uses all cores and the full bandwidth for the model download
OUTS = [OUT_NAME if s == "public_test" else f"{OUT_NAME}_{s}" for s in RUN]  # the output folders, as use() names them
REMOTE = bool(SYNC) and re.match(r"[\w.-]{2,}:", SYNC) is not None  # "gdrive:ZOLA_data" is rclone; "G:/My Drive/..." is a folder
pushed = [0.0]


def get_rclone():
    """The current rclone build, downloaded into DRIVE/rclone: no install and no admin rights needed."""
    import platform
    import urllib.request
    import zipfile

    name = "rclone.exe" if platform.system() == "Windows" else "rclone"
    home = Path(DRIVE) / "rclone"
    found = [p for p in home.rglob(name) if p.is_file()]
    if not found:
        home.mkdir(parents=True, exist_ok=True)
        osname = {"Windows": "windows", "Linux": "linux", "Darwin": "osx"}[platform.system()]
        arch = "arm64" if platform.machine().lower() in ("arm64", "aarch64") else "amd64"
        print("downloading rclone", flush=True)
        req = urllib.request.Request(f"https://downloads.rclone.org/rclone-current-{osname}-{arch}.zip", headers={"User-Agent": "notebook-fetch"})  # Python's own agent gets 403
        with urllib.request.urlopen(req, timeout=300) as r, open(home / "rclone.zip", "wb") as f:
            shutil.copyfileobj(r, f)
        with zipfile.ZipFile(home / "rclone.zip") as z:
            z.extractall(home)
        (home / "rclone.zip").unlink()
        found = [p for p in home.rglob(name) if p.is_file()]
    found[0].chmod(0o755)
    return str(found[0])


def rclone(*args, timeout=3600):
    global RCLONE
    RCLONE = RCLONE or shutil.which("rclone") or get_rclone()
    r = subprocess.run([RCLONE, *args], capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=timeout)
    if r.returncode:
        raise RuntimeError(f"rclone {args[0]} failed: {r.stderr.strip()[-800:]}")
    return r.stdout


def copy(src, dst):
    """A whole-file copy that keeps the modification time; a cut copy never replaces the old file."""
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + ".part")
    shutil.copy2(src, tmp)
    os.replace(tmp, dst)


def newer(src, dst):
    return src.is_file() and (not dst.exists() or src.stat().st_mtime > dst.stat().st_mtime + 1 or src.stat().st_size != dst.stat().st_size)


def pull():
    """From SYNC: the input files DRIVE lacks, and every file of the output folders that is newer there (traces, checkpoint)."""
    if REMOTE:
        remote = SYNC.split(":")[0] + ":"
        assert remote in rclone("listremotes").split(), f"rclone has no remote {remote}: set it up as described at the top (rclone config, or copy rclone.conf)"
        try:
            names = rclone("lsf", SYNC, timeout=300).splitlines()
        except RuntimeError as e:
            if "not found" not in str(e):
                raise
            names = []  # the folder does not exist yet: push() creates it
        for p in INPUTS:
            if not os.path.exists(p) and os.path.basename(p) in names:
                print(f"copying {os.path.basename(p)} from {SYNC}", flush=True)
                rclone("copyto", f"{SYNC}/{os.path.basename(p)}", p)
        for o in OUTS:
            if o + "/" in names:
                rclone("copy", "--update", f"{SYNC}/{o}", os.path.join(DRIVE, o))
    else:
        src = Path(SYNC)
        assert src.is_dir(), f"SYNC folder {SYNC} not found: is Google Drive for desktop running and signed in, and is the path right?"
        for p in map(Path, INPUTS):
            if not p.exists() and (src / p.name).is_file():
                print(f"copying {p.name} from {SYNC}", flush=True)
                copy(src / p.name, p)
        for o in OUTS:
            for f in (src / o).glob("*") if (src / o).is_dir() else []:
                if not f.name.endswith((".part", ".tmp")) and newer(f, Path(DRIVE) / o / f.name):
                    copy(f, Path(DRIVE) / o / f.name)
    for o in OUTS:
        t = Path(DRIVE) / o / "traces.jsonl"
        if t.exists():
            print(f"{o}: {sum(1 for _ in open(t, encoding='utf-8'))} trace lines from earlier runs", flush=True)


def push(force=False):
    """The output folders to SYNC, at most every SYNC_MIN minutes unless force. A failure is printed and never stops the run."""
    if not SYNC or not force and time.time() - pushed[0] < SYNC_MIN * 60:
        return
    pushed[0] = time.time()
    try:
        for o in OUTS:
            loc = Path(DRIVE) / o
            if not loc.is_dir():
                continue
            if REMOTE:
                rclone("copy", "--update", "--exclude", "*.tmp", str(loc), f"{SYNC}/{o}")
            else:
                for f in loc.glob("*"):
                    if not f.name.endswith((".part", ".tmp")) and newer(f, Path(SYNC) / o / f.name):
                        copy(f, Path(SYNC) / o / f.name)
        print(f"outputs copied to {SYNC}", flush=True)
    except Exception as e:
        print(f"WARNING: copying the outputs to {SYNC} failed, the run goes on: {e!r}"[:900], flush=True)


if SYNC:
    pull()
assert os.path.exists(ZIP) and ("public_test" not in RUN or os.path.exists(GEMINI)) and os.path.exists(FILL), \
    f"put train+public_test.zip, raw_asr_public.json and asr_transcripts.json in {DRIVE}" + (f" or {SYNC}" if SYNC else "")
if not os.environ.get("HF_TOKEN") and os.path.isfile(TOKEN_PATH):
    os.environ["HF_TOKEN"] = open(TOKEN_PATH, encoding="utf-8").read().strip()  # read by huggingface_hub and transformers; never printed
print("Hugging Face token:", "found" if os.environ.get("HF_TOKEN") else f"none, downloads are slower (see step 3 at the top: HF_TOKEN or {TOKEN_PATH})")


In [ ]:
import json
import platform
import shutil
import subprocess
import sys
import tarfile
import urllib.error
import urllib.request
import zipfile
from pathlib import Path

if VRAM_GB is None:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout.strip().splitlines()
    assert gpu, "nvidia-smi shows no NVIDIA GPU"
    print("\n".join(gpu))
    VRAM_GB = int(gpu[0].split(",")[1]) / 1024
# (GPU memory from, quantization, slots). Weights + one MAX_LEN context per slot must fit; the context sizes are estimates,
# so on a start-up out-of-memory lower PARALLEL first
PICK = [(44, "Q8_0", 4), (30, "Q6_K", 2), (22.5, "Q4_K_M", 2), (19, "Q4_K_M", 1), (15, "Q3_K_M", 1), (11, "IQ2_M", 1)]
auto = next(((q, p) for v, q, p in PICK if VRAM_GB >= v), None)
assert auto or (QUANT != "auto" and PARALLEL != "auto"), f"{VRAM_GB:.0f} GB of GPU memory is too little for a 27B model; set QUANT, PARALLEL and GPU_LAYERS = 'auto' yourself to try anyway"
QUANT = auto[0] if QUANT == "auto" else QUANT
PARALLEL = auto[1] if PARALLEL == "auto" else PARALLEL
CHUNK = CHUNK or PARALLEL
print(f"{VRAM_GB:.0f} GB GPU memory: {QUANT}, {PARALLEL} slot(s)")
if QUANT[:3] in ("IQ2", "Q2_", "IQ3", "Q3_") and not GGUF:
    print("WARNING: below 4 bits per weight this model is clearly weaker than the 70.88 Colab run. A 24 GB card runs Q4_K_M.")

for pkg in ("transformers", "huggingface_hub", "jinja2", "hf_xet"):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers", "huggingface_hub[hf_xet]", "jinja2"], check=True)
        break

if os.environ.get("HF_TOKEN"):  # a rejected token would make every download fail, even from public repos
    from huggingface_hub import whoami

    try:
        whoami(token=os.environ["HF_TOKEN"])
        print("Hugging Face token accepted: authenticated downloads")
    except Exception as e:
        os.environ.pop("HF_TOKEN")
        print(f"WARNING: the Hugging Face token was not accepted or could not be checked ({type(e).__name__}); downloading without it")

URL = f"http://127.0.0.1:{PORT}"
LOG = Path(DRIVE) / "llama-server.log"
server = None


def api(path, body=None, timeout=60):
    """GET (no body) or POST to the llama.cpp server, parsed JSON back."""
    req = urllib.request.Request(URL + path, None if body is None else json.dumps(body).encode(), {"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read().decode("utf-8"))


def ready():
    try:
        return api("/health", timeout=5).get("status") == "ok"
    except (OSError, ValueError):  # not listening yet, or still loading (503)
        return False


def log_tail(n=40):
    return "\n".join(open(LOG, encoding="utf-8", errors="replace").read().splitlines()[-n:]) if LOG.exists() else "(no log)"


def fetch(url, dest):
    if not dest.exists():
        print("downloading", url, flush=True)
        part = dest.with_name(dest.name + ".part")
        urllib.request.urlretrieve(url, part)
        part.replace(dest)
    return dest


def llama_server():
    """Path of llama-server and the environment to start it with. Downloads the CUDA build of release LLAMA_TAG and its CUDA runtime."""
    win = platform.system() == "Windows"
    if LLAMA_SERVER:
        exe = Path(LLAMA_SERVER)
        assert exe.exists(), exe
        return exe, dict(os.environ)
    assert platform.system() in ("Windows", "Linux") and platform.machine().lower() in ("amd64", "x86_64"), "no ready-made build for this machine: install llama.cpp with CUDA and set LLAMA_SERVER"
    home = Path(DRIVE) / "llama.cpp" / LLAMA_TAG
    name = "llama-server.exe" if win else "llama-server"
    if not list(home.rglob(name)):
        home.mkdir(parents=True, exist_ok=True)
        base = f"https://github.com/ggml-org/llama.cpp/releases/download/{LLAMA_TAG}/"
        files = [f"llama-{LLAMA_TAG}-bin-win-cuda-12.4-x64.zip", "cudart-llama-bin-win-cuda-12.4-x64.zip"] if win else \
                [f"llama-{LLAMA_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz", f"cudart-llama-{LLAMA_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz"]
        for f in files:
            arc = fetch(base + f, home / f)
            if win:
                with zipfile.ZipFile(arc) as z:
                    z.extractall(home)
            else:
                with tarfile.open(arc) as t:
                    t.extractall(home)
            arc.unlink()
    exe = sorted(home.rglob(name), key=lambda p: len(p.parts))[0]
    env = dict(os.environ)
    if win:
        for dll in home.rglob("*.dll"):  # the CUDA runtime next to the server
            if dll.parent != exe.parent and not (exe.parent / dll.name).exists():
                shutil.copy2(dll, exe.parent)
    else:
        exe.chmod(0o755)
        libs = sorted({str(p.parent) for p in home.rglob("*.so*")})
        env["LD_LIBRARY_PATH"] = os.pathsep.join(libs + [env.get("LD_LIBRARY_PATH", "")]).strip(os.pathsep)
    return exe, env


if ready():
    print(f"using the llama.cpp server already running on port {PORT} (stop it first if it holds another model or slot count)")
else:
    if GGUF is None:
        from huggingface_hub import hf_hub_download

        GGUF = hf_hub_download(GGUF_REPO, f"Qwen3.8-27B-{QUANT}.gguf", local_dir=str(Path(DRIVE) / "models"))  # resumes a broken download
    exe, env = llama_server()
    # -c is the total over all slots: every slot gets MAX_LEN tokens of its own
    cmd = [str(exe), "-m", str(GGUF), "-c", str(PARALLEL * MAX_LEN), "-np", str(PARALLEL), "--no-kv-unified", "-ngl", str(GPU_LAYERS),
           "--host", "127.0.0.1", "--port", str(PORT), "--seed", str(SEED), *map(str, SERVER_ARGS)]
    print(" ".join(cmd), flush=True)
    server = subprocess.Popen(cmd, stdout=open(LOG, "w", encoding="utf-8"), stderr=subprocess.STDOUT, env=env)
    t = time.time()
    while not ready():
        if server.poll() is not None or time.time() - t > START_MIN * 60:
            server.kill()
            raise RuntimeError(f"llama-server did not start (see the notes at the top). End of {LOG}:\n{log_tail()}")
        time.sleep(3)
    print(f"server ready after {(time.time() - t) / 60:.1f} min")

props = api("/props")
n_ctx = (props.get("default_generation_settings") or {}).get("n_ctx") or 0
print(f"slots {props.get('total_slots')}, context per slot {n_ctx}, model {props.get('model_path')}")
assert not n_ctx or n_ctx >= MAX_LEN, f"a slot holds {n_ctx} tokens, the run needs {MAX_LEN}: stop the server and check -c / -np"
print(subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip(), "GPU memory in use")

In [ ]:
%%writefile lifemate.py
"""LifeMate shared code: data split, prompt format, action parser, scorer.
02 trains on this file and saves it next to the model; 03 imports that copy, so the prompt always matches the weights.
Self-check: python lifemate.py <data root>"""
import collections
import json
import random
import re
import sys
import unicodedata
from datetime import datetime, timedelta
from pathlib import Path

ASK = "sys.ask_missing_info"
BOS, STOP = "", "<|im_end|>"
ROLES = {r: f"<|im_start|>{r}\n" for r in ("system", "user", "assistant")}  # plain ChatML: Qwen templates add a think block only to the last assistant turn, which would make training and inference differ
GEN = ROLES["assistant"]
MAX_RESULT_CHARS = 1500  # ponytail: hard cut on long tool results, summarise per tool if truncation ever hides an ID
HOLDOUT_APPS = {"door_lock", "learning", "library", "loyalty", "subscriptions", "household_chores", "home_security"}  # train-only apps: stands in for the unseen apps of the test splits
RELATIVE = {0: "hôm nay", 1: "ngày mai", 2: "ngày kia"}
WEEKS = ["tuần này", "tuần sau", "tuần sau nữa"]  # Monday-start weeks: train gold reads "thứ bảy tuần sau" as next week's Saturday
WEEKDAYS = ["Thứ Hai", "Thứ Ba", "Thứ Tư", "Thứ Năm", "Thứ Sáu", "Thứ Bảy", "Chủ Nhật"]
RULES = """Bạn là trợ lý giọng nói tiếng Việt. Hãy xử lý yêu cầu CUỐI CÙNG của người dùng bằng cách gọi tool, người dùng có thể đổi ý giữa chừng.
Mỗi lượt trả về đúng một dòng JSON, không viết gì khác:
- Gọi tool: {"name": "<app.tool>", "arguments": {...}}
- Kết thúc: {"done": true}
Quy tắc:
- Chỉ gọi tool trong danh sách, đúng tên và đúng kiểu tham số. Không tự đoán ID, hãy đọc bằng tool trước khi ghi. ID đã có trong hội thoại thì dùng lại, không đọc lại.
- Chỉ gọi tool thật sự cần. Đọc xong mà yêu cầu cần ghi thì phải ghi tiếp, đừng dừng ở bước đọc.
- Tham số không bắt buộc: chỉ điền khi người dùng nói rõ, không tự điền giá trị mặc định. Tool đọc không thêm bộ lọc nếu người dùng không nêu.
- Yêu cầu có điều kiện ("nếu mai mưa thì..."): đọc điều kiện trước, chỉ làm nhánh đúng. Điều kiện sai thì {"done": true}.
- Tìm thấy đúng một kết quả phù hợp thì dùng luôn. Không tìm thấy, nhiều kết quả không phân biệt được, hoặc thiếu tham số bắt buộc: làm xong các việc khác rồi gọi sys.ask_missing_info với tool_name là tool định gọi và params là tên các tham số còn thiếu, sau đó kết thúc.
- Không app nào làm được, người dùng tự huỷ ("à thôi khỏi"), phủ định ("đừng tắt"), hoặc chỉ trò chuyện: {"done": true} ngay, không gọi tool.
- Thời gian dùng ISO-8601 đủ giây và múi giờ, ví dụ 2026-11-11T05:30:00+07:00. Tra ngày trong bảng 14 ngày tới, không tự tính.
- Số tiền là số nguyên đồng (2 triệu = 2000000). Số đọc từng chữ số thì chép đúng thứ tự, số điện thoại giữ dạng chuỗi.
- Văn bản tự do (tin nhắn, tiêu đề, ghi chú): viết lại ngắn gọn ý chính, giữ dấu tiếng Việt, tên riêng và con số. Từ khoá tìm kiếm chỉ là tên hoặc cụm từ người dùng nhắc."""


def load_split(root, split):
    with open(Path(root) / split / f"{split}.jsonl", encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]


def load_catalog(root, split):
    with open(Path(root) / split / f"{split}_tools_catalog.json", encoding="utf-8") as f:
        return {t["name"]: t for a in json.load(f)["apps"] for t in a["tools"]}


def used_apps(x):
    return {(c["arguments"]["tool_name"] if c["name"] == ASK else c["name"]).split(".")[0] for c in x["label"]["tool_calls"]}


def clips(x):
    """Audio paths of a sample: the final request plus every user turn of the history."""
    return [x["audio"]] + [t["audio"] for t in x["history"] if t["role"] == "user"]


def gold_texts(x):
    return {x["audio"]: x["audio_text"]} | {t["audio"]: t["audio_text"] for t in x["history"] if t["role"] == "user"}


def split_rows(rows):
    """(train, val_unseen_apps, val_seen_apps). Final run: train on all rows instead."""
    train, unseen, seen = [], [], []
    for i, x in enumerate(rows):
        (unseen if used_apps(x) & HOLDOUT_APPS else seen if i % 12 == 0 else train).append(x)
    return train, unseen, seen


def render_tool(t):
    """Keeps kind, returns and x-ref: they say what is a read and which read supplies an ID, which an unseen app needs."""
    p = t["parameters"]
    o = {"name": t["name"], "kind": t["kind"], "description": t["description"]}
    if p.get("properties"):
        o["parameters"] = {k: {a: b for a, b in v.items() if a != "x-match"} for k, v in p["properties"].items()}
    for k in ("required", "x-one-of"):
        if p.get(k):
            o[k] = p[k]
    if t.get("returns"):
        o["returns"] = t["returns"]
    return json.dumps(o, ensure_ascii=False, separators=(",", ":"))


def system_prompt(ctx, tools):
    """Context stays in the system prompt, not beside the request as in Spark ver3: history calls are training targets too, and they need the clock."""
    now = datetime.fromisoformat(ctx["current_time"])
    days = ", ".join(f"{WEEKDAYS[d.weekday()]} {d:%Y-%m-%d} ({', '.join(filter(None, [RELATIVE.get(i), WEEKS[(now.weekday() + i) // 7]]))})" for i, d in enumerate(now + timedelta(i) for i in range(14)))
    return (f"{RULES}\n\nBây giờ: {ctx['current_time']} ({WEEKDAYS[now.weekday()]})\n14 ngày tới: {days}\n"
            f"Vị trí: {json.dumps(ctx.get('location'), ensure_ascii=False)}\n"
            f"Người dùng: {json.dumps(ctx.get('user_profile'), ensure_ascii=False)}\n\nDanh sách tool:\n" + "\n".join(render_tool(t) for t in tools))


def asst(obj):
    return {"role": "assistant", "content": json.dumps(obj, ensure_ascii=False)}


def result_msg(result):
    return {"role": "user", "content": "KẾT QUẢ: " + json.dumps(result, ensure_ascii=False)[:MAX_RESULT_CHARS]}


def build_messages(item, texts, catalog, rng=None):
    """texts: {audio path: transcript}. rng shuffles the tools, so the model cannot learn a position. Roles strictly alternate."""
    ctx = item["context"]
    names = [t for a in ctx["installed_apps"].values() for t in a["tools"]] + [ASK]
    if rng:
        rng.shuffle(names)
    msgs = [{"role": "system", "content": system_prompt(ctx, [catalog[n] for n in names])}]
    for turn in item["history"]:
        if turn["role"] == "user":
            msgs.append({"role": "user", "content": texts[turn["audio"]]})
            continue
        results = turn.get("tool_results") or []
        for i, c in enumerate(turn.get("tool_calls") or []):
            msgs += [asst({"name": c["name"], "arguments": c["arguments"]}), result_msg(results[i] if i < len(results) else {"ok": True})]
        msgs.append(asst({"done": True, "text": turn.get("text", "")}))
    msgs.append({"role": "user", "content": texts[item["audio"]]})
    return msgs


def gold_steps(x):
    """Gold trajectory after the final user turn: call, result, ..., done."""
    msgs = []
    for c, r in zip(x["label"]["tool_calls"], x["label"]["tool_results"]):
        msgs += [asst({"name": c["name"], "arguments": c["arguments"]}), result_msg(r)]
    return msgs + [asst({"done": True})]


def parse_action(text):
    """(name, arguments), or None for done / unparsable output (both end the turn)."""
    try:
        o = json.JSONDecoder().raw_decode(text[text.index("{"):])[0]
    except ValueError:
        return None
    if not isinstance(o.get("name"), str):
        return None
    return o["name"], o["arguments"] if isinstance(o.get("arguments"), dict) else {}


def tokens(s):
    return re.findall(r"[^\W_]+", unicodedata.normalize("NFC", str(s).lower()))  # word tokens, so "nhé." equals "nhé". The official tokeniser is unconfirmed


def token_f1(a, b):
    a, b = tokens(a), tokens(b)
    c = sum((collections.Counter(a) & collections.Counter(b)).values())
    return 2 * c / (len(a) + len(b)) if c else 0.0


def same(a, b):
    return json.dumps(a, sort_keys=True) == json.dumps(b, sort_keys=True)  # type-strict: 1, 1.0 and true all differ


def call_match(p, g, catalog):
    pa, ga = p.get("arguments") or {}, g.get("arguments") or {}
    if p["name"] != g["name"] or set(pa) != set(ga):
        return False
    props = catalog.get(g["name"], {}).get("parameters", {}).get("properties", {})
    return all(token_f1(pa[k], v) > 0.6 if props.get(k, {}).get("x-match") == "fuzzy" else same(pa[k], v) for k, v in ga.items())


def matches(pred, gold, catalog):
    """Size of the maximum one-to-one matching (Kuhn), which a greedy pass can undercount."""
    edges = [[j for j, g in enumerate(gold) if call_match(p, g, catalog)] for p in pred]
    owner = {}

    def augment(i, seen):
        for j in edges[i]:
            if j not in seen:
                seen.add(j)
                if j not in owner or augment(owner[j], seen):
                    owner[j] = i
                    return True
        return False

    return sum(augment(i, set()) for i in range(len(pred)))


def score(pred, gold, pred_hash, gold_hash, catalog):
    """Official formula from the description."""
    if not gold:
        return float(not pred)
    if pred_hash != gold_hash:
        return 0.0
    m = matches(pred, gold, catalog)
    if not m:
        return 0.0
    precision, recall = m / len(pred), m / len(gold)
    return 2 * precision * recall / (precision + recall)


def render(msgs):
    return "".join(f"{ROLES[m['role']]}{m['content']}{STOP}\n" for m in msgs)


def prompt_text(msgs):
    """Inference prompt, rendered here so that training and inference always see the same text."""
    return BOS + render(msgs) + GEN


def targets(msgs, first_target, history_calls):
    """Indices of the assistant turns to learn: every turn from first_target, plus the tool calls of history turns if history_calls.
    History replies ({"done": true, "text": ...}) stay unlabelled, since the final turn never writes text."""
    return {i for i, m in enumerate(msgs) if m["role"] == "assistant" and (i >= first_target or history_calls and m["content"].startswith('{"name"'))}


def mask_apps(msgs, apps, rng):
    """Training only: every installed app gets a made-up name in every turn, so the model has to read the tool list instead of
    remembering app names (function masking, as in Hammer, arXiv 2410.04587). Train results and transcripts never hold `app.`, checked."""
    new = {a: "".join(rng.choices("abcdefghijklmnopqrstuvwxyz", k=rng.randint(4, 9))) for a in apps}
    pat = re.compile(rf"(?<![\w@.])({'|'.join(map(re.escape, new))})\.(?=[a-z_])")
    return [m | {"content": pat.sub(lambda g: new[g.group(1)] + ".", m["content"])} for m in msgs]


def encode(tok, msgs, labelled):
    """Tokenise one conversation, labelling only the assistant turns whose index is in labelled."""
    text, spans = BOS, []
    for i, m in enumerate(msgs):
        start = len(text) + len(ROLES[m["role"]])
        text += render([m])
        if i in labelled:
            spans.append((start, len(text) - 1))  # content + end token, without the newline
    enc = tok(text, add_special_tokens=False, return_offsets_mapping=True)
    labels = [t if any(a <= s and e <= b for a, b in spans) else -100 for t, (s, e) in zip(enc["input_ids"], enc["offset_mapping"])]
    return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"], "labels": labels}


def demo(root):
    sys.path.insert(0, str(Path(root) / "train"))
    from zaic_tools import ToolEnv

    rows, catalog = load_split(root, "train"), load_catalog(root, "train")
    for x in rows[:200]:  # gold replay must score full marks, otherwise the scorer or the data path is broken
        env = ToolEnv("train", x["id"], root=root)
        gold = x["label"]["tool_calls"]
        results = [env.call(c["name"], c["arguments"]) for c in gold]
        assert results == x["label"]["tool_results"], x["id"]
        assert score(env.get_tool_calls(), gold, "h", "h", catalog) == 1.0, x["id"]

    g = [{"name": "sms.send", "arguments": {"contact_id": "c_1", "content": "mai họp lúc chín giờ"}}]
    near = [{"name": "sms.send", "arguments": {"contact_id": "c_1", "content": "Mai họp lúc chín giờ nhé"}}]
    extra = g + [{"name": "contacts.search", "arguments": {"query": "x"}}]
    wrong_id = [{"name": "sms.send", "arguments": {"contact_id": "c_2", "content": "mai họp lúc chín giờ"}}]
    assert score(near, g, "a", "a", catalog) == 1.0
    assert score(g, g, "a", "b", catalog) == 0.0
    assert score(wrong_id, g, "a", "a", catalog) == 0.0
    assert score([], g, "a", "a", catalog) == 0.0
    assert score(g, [], "a", "a", catalog) == 0.0
    assert score([], [], "x", "y", catalog) == 1.0
    assert abs(score(extra, g, "a", "a", catalog) - 2 / 3) < 1e-9
    assert token_f1("mai họp nhé.", "Mai họp, nhé") == 1.0
    fz = {"t.x": {"parameters": {"properties": {"q": {"x-match": "fuzzy"}}}}}
    assert not call_match({"name": "t.x", "arguments": {"q": "a b", "n": 1}}, {"name": "t.x", "arguments": {"q": "a b"}}, fz)  # extra key
    assert not call_match({"name": "t.x", "arguments": {"n": 1}}, {"name": "t.x", "arguments": {"n": True}}, fz)  # type
    q = [{"name": "t.x", "arguments": {"q": s}} for s in ("một hai ba bốn", "hai ba năm sáu", "một hai ba năm")]
    assert matches(q[:2], [q[2], q[0]], fz) == 2  # the first pred fits both golds, the second only the first: greedy finds 1

    assert parse_action('{"name": "a.b", "arguments": {"x": 1}} rác') == ("a.b", {"x": 1})
    assert parse_action('rác {"name": "a.b"}') == ("a.b", {})
    assert parse_action('{"done": true}') is None
    assert parse_action("không có json") is None
    assert parse_action('{"name": ') is None

    sp = system_prompt({"current_time": "2026-09-24T09:00:00+07:00"}, [])  # a Thursday
    assert "Thứ Năm 2026-09-24 (hôm nay, tuần này)" in sp and "Chủ Nhật 2026-09-27 (tuần này)" in sp and "Thứ Bảy 2026-10-03 (tuần sau)" in sp, sp

    train, unseen, seen = split_rows(rows)
    assert len(train) + len(unseen) + len(seen) == len(rows)
    assert not any(used_apps(x) & HOLDOUT_APPS for x in train)
    for x in rows:
        msgs = build_messages(x, gold_texts(x), catalog) + gold_steps(x)
        roles = [m["role"] for m in msgs[1:]]
        assert all(a != b for a, b in zip(roles, roles[1:])), x["id"]
        first = len(msgs) - 2 * len(x["label"]["tool_calls"]) - 1
        assert msgs[first - 1]["content"] == x["audio_text"], x["id"]
        hist = [c for t in x["history"] for c in t.get("tool_calls") or []]
        assert len(targets(msgs, first, True)) == len(hist) + len(x["label"]["tool_calls"]) + 1 and len(targets(msgs, first, False)) == len(x["label"]["tool_calls"]) + 1, x["id"]
        m = mask_apps(msgs, x["context"]["installed_apps"], random.Random(0))
        called = [json.loads(t["content"])["name"] for t in m[first:] if t["role"] == "assistant" and t["content"].startswith('{"name"')]
        assert all(f'"name":"{n}"' in m[0]["content"] for n in called) and not any(f'"name":"{a}.' in m[0]["content"] for a in x["context"]["installed_apps"]), x["id"]
    print("lifemate self-check ok:", len(train), "train,", len(unseen), "unseen-app val,", len(seen), "seen-app val")


if __name__ == "__main__":
    demo(sys.argv[1])

In [ ]:
import glob
import importlib
import json
import math
import sys
import zipfile
from collections import Counter
from pathlib import Path

DATA = Path(DRIVE) / "data"
if not all(glob.glob(f"{DATA}/**/{s}/{s}.jsonl", recursive=True) for s in ("train", "public_test")):
    with zipfile.ZipFile(ZIP) as z:  # both splits, no audio: the agent reads transcripts, the examples come from train
        z.extractall(DATA, [m for m in z.namelist() if not m.endswith(".wav")])
ROOT = Path(glob.glob(f"{DATA}/**/public_test/public_test.jsonl", recursive=True)[0]).parents[1]
sys.path.insert(0, ".")  # lifemate.py from this notebook
import lifemate as lm

train_rows, train_catalog = lm.load_split(ROOT, "train"), lm.load_catalog(ROOT, "train")
fit, val_unseen, val_seen = lm.split_rows(train_rows)


def load_texts(path, rows, split):
    """{"<split>/<clip path>": text}. Also takes transcripts keyed by sample id: <id> is the final request, <id>_h<N> is history[N]."""
    t = json.load(open(path, encoding="utf-8"))
    if all("/" in k for k in t):
        return t
    clip = {x["id"]: x["audio"] for x in rows} | {f"{x['id']}_h{n}": h["audio"] for x in rows for n, h in enumerate(x["history"]) if h["role"] == "user"}
    return {f"{split}/{clip[k]}": v for k, v in t.items() if k in clip}


def grams(text):
    t = lm.tokens(text)
    return t + [a + " " + b for a, b in zip(t, t[1:])]


def vec(text):
    """Unit TF-IDF vector over words and word pairs."""
    w = {g: n * IDF[g] for g, n in Counter(grams(text)).items() if g in IDF}
    norm = math.sqrt(sum(v * v for v in w.values())) or 1.0
    return {g: v / norm for g, v in w.items()}


def use(split):
    """Makes split the current one: its rows, catalog and zaic_tools (each split ships its own, R-ENV-6), its transcripts, output folder,
    and the train rows the examples and argument habits come from: all of train for public_test, only the non-validation rows for the
    train check, so a checked row never sees itself or its held-out apps."""
    global SPLIT, rows, catalog, ToolEnv, texts, alt, pool, OUT, IDF, DOCS, CALLS, USED
    SPLIT = split
    for m in [m for m in sys.modules if m.split(".")[0] == "zaic_tools"]:
        del sys.modules[m]
    sys.path[:] = [p for p in sys.path if p not in {str(ROOT / s) for s in ("train", "public_test")}]
    sys.path.insert(0, str(ROOT / split))
    importlib.invalidate_caches()
    ToolEnv = importlib.import_module("zaic_tools").ToolEnv
    catalog = lm.load_catalog(ROOT, split)
    if split == "train":
        val = val_unseen + val_seen
        rows, pool = val[::max(1, len(val) // TRAIN_CHECK)][:TRAIN_CHECK], fit
        texts, alt = {f"{split}/{k}": v for x in rows for k, v in lm.gold_texts(x).items()}, {}
    else:
        rows, pool = lm.load_split(ROOT, split), train_rows
        gem = load_texts(GEMINI, rows, split)
        fill = load_texts(FILL, rows, split) if os.path.exists(FILL) else {}
        texts = fill | gem  # Gemini wins wherever it has the clip
        alt = {k: fill[k] for k in gem if k in fill and lm.token_f1(fill[k], gem[k]) < ALT_BELOW}
    OUT = Path(DRIVE) / (OUT_NAME if split == "public_test" else f"{OUT_NAME}_{split}")
    OUT.mkdir(parents=True, exist_ok=True)
    need = [f"{split}/{c}" for x in rows for c in lm.clips(x)]
    miss = [k for k in need if k not in texts]
    assert not miss, f"no text for {len(miss)} clips, e.g. {miss[:3]}: put 01c's asr_transcripts.json at {FILL}"
    df = Counter(g for x in pool for g in set(grams(x["audio_text"])))
    IDF = {g: math.log(1 + len(pool) / n) for g, n in df.items()}
    DOCS = [(x, vec(x["audio_text"]), {c["name"] for c in x["label"]["tool_calls"]}) for x in pool]
    CALLS = Counter(c["name"] for x in pool for c in x["label"]["tool_calls"])
    USED = Counter((c["name"], a) for x in pool for c in x["label"]["tool_calls"] for a in c["arguments"])
    two = sum(split + "/" + x["audio"] in alt for x in rows)
    print(f"{split}: {len(rows)} samples, {len(need)} clips, {two} final requests with a second transcript, {len(pool)} train rows for examples -> {OUT}")


def pick(x, text):
    """Up to DEMOS train rows. First the most similar request; then, for the installed tools that the 40 most similar rows use most, the
    most similar row that calls each (up to 4 tools), so the examples show gold's way of calling the tools this request probably needs;
    then the next most similar rows, at most two with the same set of tools. Similarity is scaled down when a row's tools are not installed here."""
    q = vec(text)
    have = {t for a in x["context"]["installed_apps"].values() for t in a["tools"]} | {lm.ASK}
    scored = sorted(((sum(w * d.get(g, 0.0) for g, w in q.items()) * (0.5 + 0.5 * (len(tools & have) / len(tools) if tools else 1.0)), i) for i, (_, d, tools) in enumerate(DOCS)), reverse=True)
    scored = [(s, i) for s, i in scored if s > 0 and DOCS[i][0]["id"] != x["id"]]
    if not scored:
        return []
    out, votes = [scored[0][1]], Counter()
    for s, i in scored[:40]:
        for t in DOCS[i][2]:
            votes[t] += s
    for t, _ in votes.most_common():
        if len(out) >= min(DEMOS, 5):
            break
        if t in have and not any(t in DOCS[i][2] for i in out):
            out += [i for _, i in scored if t in DOCS[i][2] and DOCS[i][2] <= have and i not in out][:1]
    taken = Counter(tuple(sorted(DOCS[i][2])) for i in out)
    for _, i in scored:
        key = tuple(sorted(DOCS[i][2]))
        if len(out) >= DEMOS:
            break
        if i not in out and taken[key] < 2:
            taken[key] += 1
            out.append(i)
    return [DOCS[i][0] for i in out[:DEMOS]]


def demo_text(y):
    """One train row as text: the last turns of its history in short, its request, then every gold call with its result cut short."""
    short = lambda o, n: json.dumps(o, ensure_ascii=False)[:n]
    lines = []
    for t in y["history"][-4:]:
        if t["role"] == "user":
            lines.append("Người dùng (lượt trước): " + t["audio_text"][:200])
        else:
            lines += ["Trợ lý (lượt trước): " + short({"name": c["name"], "arguments": c["arguments"]}, 200) for c in (t.get("tool_calls") or [])[-3:]]
    lines.append("Người dùng: " + y["audio_text"])
    for c, r in zip(y["label"]["tool_calls"], y["label"]["tool_results"]):
        lines += ["Trợ lý: " + json.dumps({"name": c["name"], "arguments": c["arguments"]}, ensure_ascii=False), "KẾT QUẢ: " + short(r, DEMO_RESULT_CHARS)]
    return "\n".join(lines + ['Trợ lý: {"done": true}'])


def demo_block(demos):
    return "".join(f"\n\nVí dụ mẫu {n}:\n{demo_text(y)}" for n, y in enumerate(demos, 1))


def hints(x):
    """Per installed tool that train gold calls at least 8 times: optional parameters gold fills in at most 5% of its calls, and in at least 60%."""
    lines = []
    for name in sorted(t for a in x["context"]["installed_apps"].values() for t in a["tools"]):
        p = train_catalog.get(name, {}).get("parameters", {})
        fixed = set(p.get("required", [])) | {k for grp in p.get("x-one-of") or [] for k in grp}
        opt = [k for k in p.get("properties", {}) if k not in fixed]
        if CALLS[name] < 8 or not opt:
            continue
        rare = [k for k in opt if USED[(name, k)] <= 0.05 * CALLS[name]]
        often = [k for k in opt if USED[(name, k)] >= 0.6 * CALLS[name]]
        if rare or often:
            lines.append(f"- {name}: " + "; ".join(filter(None, ["hầu như không bao giờ điền " + ", ".join(rare) if rare else "", "thường điền " + ", ".join(often) if often else ""])))
    return "\n\nThói quen tham số tuỳ chọn trong lời giải chuẩn (làm theo, trừ khi người dùng yêu cầu rõ đúng tham số đó):\n" + "\n".join(lines) if lines else ""


use(RUN[0])

In [ ]:
from concurrent.futures import ThreadPoolExecutor

from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(TOKENIZER_ID)  # Qwen's tokenizer and chat template; the weights stay in the llama.cpp server
STOP = tok.encode(lm.STOP, add_special_tokens=False)
END = tok.convert_tokens_to_ids("</think>")
CLOSE = tok.encode("\n</think>\n\n", add_special_tokens=False)
assert len(STOP) == 1 and tok.encode("</think>", add_special_tokens=False) == [END], (STOP, END)
ENDS = set(STOP) | ({tok.eos_token_id} - {None})  # a turn that stopped inside the thought ends with one of these; dropped before the thought is closed
probe = "Đặt báo thức 6 giờ sáng mai.\n</think>\n\n" + lm.STOP
assert api("/tokenize", {"content": probe, "parse_special": True})["tokens"] == tok.encode(probe, add_special_tokens=False), \
    "the GGUF's vocabulary is not the tokenizer's: token-id prompts would be garbage. Use a GGUF made from Qwen/Qwen3.8-27B"
SPEED = Counter()  # tokens and milliseconds reported by the server


def prompt_ids(msgs):
    """Qwen3.8's own chat template, so thinking and reasoning_effort work as the model was trained."""
    return tok.encode(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=THINK, reasoning_effort=EFFORT), add_special_tokens=False)


def complete(ids, n):
    """(generated token ids, cut at n tokens?, server timings) for a token-id prompt. Greedy. The prompt cache makes a sample's next step
    and the continuation after a closed thought pay only for their new tokens."""
    body = {"prompt": ids, "n_predict": n, "temperature": 0, "top_k": 1, "seed": SEED, "cache_prompt": True, "return_tokens": True}
    for wait in (5, 20, 60, None):
        try:
            r = api("/completion", body, timeout=3600)
            return list(r["tokens"]), r.get("stop_type") == "limit", r.get("timings") or {}
        except urllib.error.HTTPError as e:
            raise RuntimeError(f"llama-server answered {e.code}: {e.read()[:500]!r}") from None
        except OSError as e:  # connection refused or reset, timeout
            if wait is None or (server is not None and server.poll() is not None):
                raise RuntimeError(f"llama-server is gone ({e!r}). End of {LOG}:\n{log_tail()}") from None
            time.sleep(wait)


def one(ids):
    """(token ids of thought + answer, closed, timings) for one prompt: the thought within its budget, closed for the model if needed, then the answer."""
    budget = THINK_TOKENS if THINK else ANSWER_TOKENS
    g, limit, t = complete(ids, max(1, min(budget, MAX_LEN - len(ids) - ANSWER_TOKENS - 8)))
    closed, times = 0, [t]
    if THINK and (limit or END not in g):  # token ids, not text, so the continuation sees exactly what was generated
        cut = END not in g
        while g and g[-1] in ENDS:
            g.pop()
        g += CLOSE if cut else []
        more, _, t = complete(ids + g, ANSWER_TOKENS)
        g, closed = g + more, ((1 if limit else 2) if cut else 0)
        times.append(t)
    while g and g[-1] in ENDS:
        g.pop()
    return g, closed, times


def generate(convs):
    """[(thought, answer, closed)] per conversation, PARALLEL at a time. closed: 0; 1 for a thought cut at THINK_TOKENS and 2 for a turn that
    ended inside the thought (both are closed for the model, then answered); 3 for a prompt that leaves no room for an answer (not generated)."""
    ids = [prompt_ids(m) for m in convs]
    res = [("", "", 3)] * len(ids)
    fits = [k for k, i in enumerate(ids) if len(i) <= MAX_LEN - ANSWER_TOKENS - 64]
    with ThreadPoolExecutor(PARALLEL) as pool_:
        outs = list(pool_.map(one, [ids[k] for k in fits]))
    for k, (g, closed, times) in zip(fits, outs):  # decoded here, not in the threads: the tokenizer is not thread-safe
        res[k] = (*tok.decode(g).rpartition("</think>")[::2], closed)
        for t in times:
            SPEED.update(gen=t.get("predicted_n", 0), gen_ms=t.get("predicted_ms", 0), prompt=t.get("prompt_n", 0), prompt_ms=t.get("prompt_ms", 0))
    return res


def messages(x):
    """(conversation, ids of the examples used): lifemate's conversation, the second transcript on the final request, and EXTRA_RULES,
    argument habits and examples appended to the system prompt. Examples are dropped from the least similar until the prompt fits."""
    key = f"{SPLIT}/{x['audio']}"
    m = lm.build_messages(x, {c: texts[f"{SPLIT}/{c}"] for c in lm.clips(x)}, catalog)
    if ALT_ASR and key in alt:
        m[-1] = m[-1] | {"content": m[-1]["content"] + ALT_NOTE.format(alt[key])}
    extra = EXTRA_RULES + (hints(x) if ARG_HINTS else "")
    demos = pick(x, texts[key]) if DEMOS else []
    build = lambda d: [m[0] | {"content": m[0]["content"] + extra + demo_block(d)}] + m[1:]
    while demos and len(prompt_ids(build(demos))) > FIRST_PROMPT_MAX:
        demos.pop()
    return build(demos), [y["id"] for y in demos]


sample, used = messages(rows[0])
print(sample[0]["content"][len(lm.build_messages(rows[0], {c: texts[f"{SPLIT}/{c}"] for c in lm.clips(rows[0])}, catalog)[0]["content"]):][:6000])
print("...\n" + tok.apply_chat_template(sample, tokenize=False, add_generation_prompt=True, enable_thinking=THINK, reasoning_effort=EFFORT)[-500:])
sizes = [(len(prompt_ids(m)), len(d)) for m, d in map(messages, rows)]
print(f"ready after {(time.time() - T0) / 60:.1f} min. First prompts: longest {max(s[0] for s in sizes)} tokens, mean {sum(s[0] for s in sizes) / len(sizes):.0f}; "
      f"examples per sample {sum(s[1] for s in sizes) / len(sizes):.1f}, {sum(s[1] < DEMOS for s in sizes)} samples with fewer than {DEMOS}")

In [ ]:
def missing(name, args, installed):
    """The ask gold would make for an installed tool that misses a required or one-of parameter, else None. From docker/predict.py."""
    tool = catalog.get(name)
    if tool is None or name.split(".")[0] not in installed:
        return None
    p = tool.get("parameters", {})
    empty = lambda k: args.get(k) in (None, "", [], {})  # zaic_tools counts these as missing
    need = [k for k in p.get("required", []) if empty(k)]
    if p.get("x-one-of") and all(any(empty(k) for k in grp) for grp in p["x-one-of"]):
        need.append(p["x-one-of"][0][0])
    return {"tool_name": name, "params": need} if need else None


def step(s, answer):
    """One model answer on the sample's env; False when the sample stops. The branches of docker/predict.py's run()."""
    env, msgs = s["env"], s["msgs"]
    s["raw"].append(answer)
    action = lm.parse_action(answer)
    if action is None:  # parse_action returns None for both
        s["stop"] = "done" if '"done"' in answer else "unparsable"
        return False
    call = {"name": action[0], "arguments": action[1]}
    said = json.JSONDecoder().raw_decode(answer[answer.index("{"):])[0]
    check = env.validate(*action)
    if not check["ok"]:  # validate is not logged: a call we never make costs no U penalty
        ask = missing(*action, s["item"]["context"]["installed_apps"])
        if ask and env.validate(lm.ASK, ask)["ok"]:
            env.call(lm.ASK, ask)
            s["stop"] = "repaired"
            return False
        if not s["retried"]:  # one more try with the error as the result
            s["retried"] = True
            msgs += [lm.asst(said), lm.result_msg(check)]
            return True
        s["stop"] = "invalid"
        return False
    calls = env.get_tool_calls()
    if calls.count(call) >= 2 or catalog.get(call["name"], {}).get("kind") == "read" and [c["name"] for c in calls].count(call["name"]) >= 2:
        s["stop"] = "repeat"
        return False
    result = env.call(*action)
    msgs += [lm.asst(said), lm.result_msg(result)]
    if LIST_ON_EMPTY and "query" in action[1] and not catalog[call["name"]]["parameters"].get("required") and any(v == [] for v in result.values()):  # misheard query: list everything once
        msgs += [lm.asst({"name": call["name"], "arguments": {}}), lm.result_msg(env.call(call["name"], {}))]
    if call["name"] == lm.ASK:  # gold always ends here
        s["stop"] = "ask"
        return False
    return True


KEEP = ("msgs", "raw", "thoughts", "capped", "unclosed", "retried", "demos")  # what a checkpoint carries of a running sample


def new(x):
    msgs, demos = messages(x)
    s = {"item": x, "env": ToolEnv(SPLIT, x["id"], root=ROOT), "msgs": msgs, "demos": demos, "request": msgs[-1]["content"],
         "raw": [], "thoughts": [], "capped": 0, "unclosed": 0, "stop": "max_steps", "retried": False, "saved": False}  # exactly one env per sample
    c = resume.pop(x["id"], None)
    if c:  # mid-sample at the last checkpoint: its env died with the runtime, so replay its calls on the new one and carry on
        for call in c["calls"]:
            s["env"].call(call["name"], call["arguments"])
        assert s["env"].get_tool_calls() == c["calls"], x["id"]
        s |= {k: c[k] for k in KEEP}
    return s


def checkpoint(live):
    """The samples still running, after every round, so a disconnect loses no step. Written to a temp file first: a cut write leaves the old one."""
    tmp = CKPT.with_suffix(".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump([{"id": s["item"]["id"], "calls": s["env"].get_tool_calls()} | {k: s[k] for k in KEEP} for s in live], f, ensure_ascii=False)
    os.replace(tmp, CKPT)


def save(s, f):
    if s["saved"]:
        return
    env, x = s["env"], s["item"]
    f.write(json.dumps({"id": x["id"], "tool_calls": env.get_tool_calls(), "state_hash": env.get_state_hash(), "stop": s["stop"], "capped": s["capped"], "unclosed": s["unclosed"],
                        "raw": s["raw"], "thoughts": s["thoughts"], "request": s["request"],
                        "demos": s["demos"]}, ensure_ascii=False) + "\n")
    f.flush()
    s["saved"] = True


left = lambda: T0 + (HOURS * 60 - MARGIN_MIN) * 60 - time.time()


def run(out):
    """Every sample of rows not finished in out/traces.jsonl, in chunks, until the time margin. Rerun-safe: finished samples are
    kept, checkpointed ones replay their calls and continue."""
    global CKPT, resume  # read by new() and checkpoint()
    traces, CKPT = out / "traces.jsonl", out / "ckpt.json"
    old = {}
    if traces.exists():
        for line in map(json.loads, open(traces, encoding="utf-8")):
            old[line["id"]] = line  # the last line of a sample wins
    todo = [x for x in rows if old.get(x["id"], {}).get("stop", "not_run") == "not_run"]
    resume = {c["id"]: c for c in (json.load(open(CKPT, encoding="utf-8")) if CKPT.exists() else []) if c["id"] in {x["id"] for x in todo}}
    todo.sort(key=lambda x: x["id"] not in resume)  # checkpointed samples go in the first chunk
    print(f"{len(resume)} samples resume from {CKPT}")
    print(f"{len(rows) - len(todo)} done in earlier runs, {len(todo)} to run, {left() / 60:.0f} min before the margin")
    with open(traces, "a", encoding="utf-8") as f:
        t_run = time.time()
        for start in range(0, len(todo), CHUNK):
            if left() <= 0:
                print("time limit (HOURS): the rest become empty answers below; run all again to continue")
                break
            t, live = time.time(), [new(x) for x in todo[start:start + CHUNK]]
            try:
                while live and left() > 0:
                    going = []
                    for s, (thought, answer, closed) in zip(live, generate([s["msgs"] for s in live])):
                        if closed == 3:  # the conversation no longer fits the context: keep the calls made so far
                            s["stop"] = "context"
                            save(s, f)
                            continue
                        s["thoughts"].append(thought)
                        s["capped"] += closed == 1
                        s["unclosed"] += closed == 2
                        try:
                            on = step(s, answer)
                        except Exception as e:  # keep the calls already made
                            s["stop"], on = "crash", False
                            print(s["item"]["id"], repr(e), flush=True)
                        if on and len(s["raw"]) < MAX_STEPS:  # steps count across a resume
                            going.append(s)
                        else:
                            save(s, f)
                    live = going
                    checkpoint(live)
                if left() <= 0:
                    for s in live:
                        s["stop"] = "deadline"
            except BaseException:  # an engine error or a stopped cell: the calls made so far still go out
                for s in live:
                    s["stop"] = "crash"
                raise
            finally:
                for s in live:
                    save(s, f)
                checkpoint([])
            n = min(start + CHUNK, len(todo))
            if n == len(todo) or n // 16 > start // 16:  # about every 16 samples
                print(f"{n}/{len(todo)}: {(time.time() - t_run) / 60:.0f} min so far, about {(time.time() - t_run) / n * (len(todo) - n) / 60:.0f} min to go; "
                      f"generation {1000 * SPEED['gen'] / max(1, SPEED['gen_ms']):.0f} tokens/s per slot, prompt {1000 * SPEED['prompt'] / max(1, SPEED['prompt_ms']):.0f} tokens/s", flush=True)
                push()

In [ ]:
import subprocess


def finish(out):
    """Empty answers for samples never started, run summary, submission. Train: also the score against gold, kept per sample in scored.jsonl."""
    traces = out / "traces.jsonl"
    by_id = {}
    for line in map(json.loads, open(traces, encoding="utf-8")):
        by_id[line["id"]] = line
    with open(traces, "a", encoding="utf-8") as f:
        for x in rows:
            if x["id"] not in by_id:  # never started: an empty answer on an env with no call, so a later run may still do it
                by_id[x["id"]] = {"id": x["id"], "tool_calls": [], "state_hash": ToolEnv(SPLIT, x["id"], root=ROOT).get_state_hash(), "stop": "not_run"}
                f.write(json.dumps(by_id[x["id"]]) + "\n")
    lines = [by_id[x["id"]] for x in rows]
    thoughts = [t for x in lines for t in x.get("thoughts", [])]
    print("stops:", Counter(x["stop"] for x in lines).most_common())
    print(f"{sum(len(x['tool_calls']) for x in lines)} calls, {len(thoughts)} model steps, {sum(map(len, thoughts)) / max(1, len(thoughts)):.0f} thought chars per step, "
          f"{sum(x.get('capped', 0) for x in lines)} thoughts cut at THINK_TOKENS, {sum(x.get('unclosed', 0) for x in lines)} turns ended inside the thought (closed and answered)")
    asks = [c for x in lines for c in x["tool_calls"] if c["name"] == lm.ASK]
    reads = [c for x in lines for c in x["tool_calls"] if catalog.get(c["name"], {}).get("kind") == "read"]
    print(f"asks in {sum(any(c['name'] == lm.ASK for c in x['tool_calls']) for x in lines) / len(lines):.0%} of samples ({sum(any(p.endswith('_id') for p in c['arguments'].get('params', [])) for c in asks)} for an ID), "
          f"empty answers {sum(not x['tool_calls'] for x in lines) / len(lines):.0%}, reads without arguments {sum(not c['arguments'] for c in reads) / max(1, len(reads)):.0%} "
          f"(train gold: 14% ask, 7% empty, 60% of reads without arguments; v2 public: 17%, 8%, 32%)")
    sub = out / "submission.jsonl"
    with open(sub, "w", encoding="utf-8") as f:
        for x in lines:
            f.write(json.dumps({k: x[k] for k in ("id", "tool_calls", "state_hash")}, ensure_ascii=False) + "\n")
    if SPLIT != "train":
        print(subprocess.run([sys.executable, "-m", "zaic_tools", "--split", SPLIT, "--root", "..", "--validate", str(sub)], cwd=ROOT / SPLIT, env=os.environ | {"PYTHONPATH": ".", "PYTHONUTF8": "1"}, capture_output=True, text=True, encoding="utf-8", errors="replace").stdout)
        print(f"submit {sub}")
        return
    # eval only: the gold hash comes from replaying gold on a second env of the sample, as lifemate.demo does. Never in a submission run.
    unseen, per = {x["id"] for x in val_unseen}, []
    with open(out / "scored.jsonl", "w", encoding="utf-8") as f:
        for x, g in zip(lines, rows):
            genv = ToolEnv(SPLIT, g["id"], root=ROOT)
            for c in g["label"]["tool_calls"]:
                genv.call(c["name"], c["arguments"])
            same = x["state_hash"] == genv.get_state_hash()
            sc = lm.score(x["tool_calls"], g["label"]["tool_calls"], x["state_hash"], genv.get_state_hash(), catalog)
            per.append((sc, g["id"] in unseen, same or not g["label"]["tool_calls"]))
            f.write(json.dumps({"id": g["id"], "score": sc, "hash_ok": same, "unseen_app": g["id"] in unseen, "stop": x["stop"], "request": g["audio_text"],
                                "tool_calls": x["tool_calls"], "gold": g["label"]["tool_calls"]}, ensure_ascii=False) + "\n")
    mean = lambda v: 100 * sum(v) / max(1, len(v))
    print(f"train check (gold text, {len(per)} rows): score {mean([p[0] for p in per]):.2f}, unseen apps {mean([p[0] for p in per if p[1]]):.2f}, seen apps {mean([p[0] for p in per if not p[1]]):.2f}")
    print(f"  perfect {sum(p[0] == 1 for p in per)}, right final state but calls differ {sum(p[2] and p[0] < 1 for p in per)} (they keep {mean([p[0] for p in per if p[2] and p[0] < 1]):.0f}% on average), "
          f"wrong final state {sum(not p[2] for p in per)}. Per sample: {out / 'scored.jsonl'}")


for split in RUN:
    if left() <= 0:
        print(f"time margin reached before {split}: run all again to continue")
        break
    use(split)
    print(f"\n== {split}", flush=True)
    t = time.time()
    try:
        run(OUT)
        finish(OUT)
    finally:
        push(force=True)
    print(f"{split}: {(time.time() - t) / 60:.1f} min", flush=True)
print(f"{(time.time() - T0) / 3600:.2f} h in total")

In [ ]:
# Frees the GPU. Skip this cell to keep the model loaded for another run.
if server is not None:
    server.terminate()
    print("llama-server stopped")
